In [1]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.


C:\Users\anyab\OneDrive\Documents\ITMO\5 семестр\ОАД\.venv\Scripts\python.exe: No module named pip


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer


In [ ]:
df = pd.read_csv('Walmart.csv')

In [ ]:
df

In [ ]:
print(df.info())
print('\n')
print(df.isnull().sum())

In [ ]:
def iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

numeric_columns = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_columns, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[col])
    plt.title(f'{col}')

plt.tight_layout()
plt.show()

for col in numeric_columns:
    outliers, lower_bound, upper_bound = iqr(df, col)
    print(f"Выбросы в {col}: {len(outliers)}")
    df[col] = np.where(df[col] > upper_bound, upper_bound, 
                      np.where(df[col] < lower_bound, lower_bound, df[col]))

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format="%d-%m-%Y")
df['Store'] = df['Store'].astype('category')
df['Holiday_Flag'] = df['Holiday_Flag'].astype('category')

print(df.dtypes)

In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df['DayOfYear'] = df['Date'].dt.dayofyear
df['IsWeekend'] = (df['Date'].dt.dayofweek >= 5).astype(int)

df['Season'] = df['Month'].map({12: 'Winter', 1: 'Winter', 2: 'Winter',
                               3: 'Spring', 4: 'Spring', 5: 'Spring',
                               6: 'Summer', 7: 'Summer', 8: 'Summer',
                               9: 'Autumn', 10: 'Autumn', 11: 'Autumn'})


df = df.sort_values(['Store', 'Date'])
df['Sales_Lag_1'] = df.groupby('Store')['Weekly_Sales'].shift(1)
df['Sales_Lag_1'] = df.groupby('Store')['Sales_Lag_1'].transform(
    lambda x: x.fillna(x.median()))
df['Sales_Lag_4'] = df.groupby('Store')['Weekly_Sales'].shift(4) 
df['Sales_Lag_4'] = df.groupby('Store')['Sales_Lag_4'].transform(
    lambda x: x.fillna(x.median()))
df['Sales_Lag_52'] = df.groupby('Store')['Weekly_Sales'].shift(52)  
df['Sales_Lag_52'] = df.groupby('Store')['Sales_Lag_52'].transform(
    lambda x: x.fillna(x.median()))



df['Sales_Mean_4'] = df.groupby('Store')['Weekly_Sales'].transform(
    lambda x: x.rolling(window=4, min_periods=1).mean())
df['Sales_Mean_12'] = df.groupby('Store')['Weekly_Sales'].transform(
    lambda x: x.rolling(window=12, min_periods=1).mean())


store_stats = df.groupby('Store').agg({
    'Weekly_Sales': ['mean', 'min', 'max'],
    'Temperature': 'mean',
    'Fuel_Price': 'mean',
    'CPI': 'mean',
    'Unemployment': 'mean'
}).round(2)
store_stats.columns = ['_'.join(col).strip() for col in store_stats.columns.values]


df = df.merge(store_stats, on='Store', how='left')


df['Sales_Trend'] = df.groupby('Store')['Weekly_Sales'].transform(
    lambda x: x.diff().rolling(window=4, min_periods=1).mean())
df


In [ ]:
features =  ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 
                   'Sales_Lag_1', 'Sales_Lag_4', 'Sales_Lag_52',
                   'Sales_Mean_4', 'Sales_Mean_12']

for feature in features:
    data = df[feature].dropna()
    
    skew_val = data.skew()
    print(f"\n {feature}:")
    print(f"   Асимметрия: {skew_val:.2f} ")
    
    plt.figure(figsize=(8, 2))
    plt.hist(data, bins=30, alpha=0.7)
    plt.title(f'{feature} (skew: {skew_val:.2f})')
    plt.show()
    
    if abs(skew_val) < 0.5:
        print("StandardScaler")
    elif abs(skew_val) > 1 :
        print("RobustScaler") 
    else:
        print("PowerTransformer")


In [ ]:
for col in numeric_features:
    df[col] = df.groupby('Store')[col].transform(
        lambda x: x.fillna(x.median()))
numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 
                   'Sales_Lag_1', 'Sales_Lag_4', 'Sales_Lag_52',
                   'Sales_Mean_4', 'Sales_Mean_12', 'Sales_Trend']

df_clean = df.dropna(subset=numeric_features)

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_clean[numeric_features])

scaled_df = pd.DataFrame(scaled_features, columns=[f'{col}_scaled' for col in numeric_features])
scaled_df.index = df_clean.index

df = pd.concat([df, scaled_df], axis=1)
print("Готово")

In [ ]:
season_dummies = pd.get_dummies(df['Season'], prefix='Season')
df = pd.concat([df, season_dummies], axis=1)

le = LabelEncoder()
df['Store_encoded'] = le.fit_transform(df['Store'])


In [ ]:
print(f"Размер: {df.shape}")
print("\nПропущенные значения:")
print(df.isnull().sum().sum())

df.to_csv('Walmart_cleaned.csv', index=False)

plt.figure(figsize=(15, 12))
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(correlation_matrix[['Weekly_Sales']].sort_values('Weekly_Sales', ascending=False), 
            annot=True, cmap='coolwarm', center=0)
plt.title('Корреляция признаков с Weekly_Sales')
plt.tight_layout()
plt.show()